# HumanEvalComm V2 — Ollama Benchmark on Colab

Runs the HumanEvalComm V2 benchmark pipeline entirely with **local Ollama models**
on a Colab GPU instance.

## How to use
1. Select **Runtime → Change runtime type → T4 GPU**.
2. Run cells **top-to-bottom**.  Each section header tells you what it does.
3. To run only one model for a quick smoke test, set `SINGLE_MODEL_SMOKE = True`
   in the **Configuration** cell.

## Default models (all fit comfortably on a free T4)

| Alias | Ollama tag | VRAM (approx) |
|---|---|---|
| `qwen25_coder_7b` | `qwen2.5-coder:7b` | ~4 GB |
| `deepseek_coder_6b` | `deepseek-coder:6.7b` | ~4 GB |
| `codegemma_7b` | `codegemma:7b` | ~4 GB |
| `starcoder2_7b` | `starcoder2:7b` | ~4 GB |
| `codellama_7b` | `codellama:7b-instruct` | ~4 GB |

> **Tip:** pulling and evaluating all five models takes a long time.
> Start with `SINGLE_MODEL_SMOKE = True` to validate the pipeline, then
> set it to `False` for the full run.


In [ ]:
#@title ⚙️ Configuration — run this first
# ── Repository ──────────────────────────────────────────────────────
UPSTREAM_REMOTE_URL = "https://github.com/ssoad/human-eval-comm.git"
UPSTREAM_REF        = "beta"
REPO_DIR            = "/content/human-eval-comm-v2"

# ── Results storage ─────────────────────────────────────────────────
# Default: local /content (lost when session ends).
# After mounting Drive (next cell) this is overridden automatically.
RESULTS_ROOT = "/content/results"

# ── Benchmark limits ────────────────────────────────────────────────
SMOKE_MAX_PROBLEMS      = 2      # problems for the quick smoke test
STANDARD_MAX_PROBLEMS   = 1000
UNFEASIBLE_MAX_PROBLEMS = 1000

# ── Ollama settings ─────────────────────────────────────────────────
LOCAL_API_BASE = "http://localhost:11434/v1"
# Per-request timeout in seconds.  Keep >=300 for local models — the
# first inference call includes model loading which can be slow.
MODEL_TIMEOUT  = 300

# ── Model list ───────────────────────────────────────────────────────
# All 5 models fit on a free T4 (16 GB VRAM) individually.
OLLAMA_MODELS = [
    {"name": "qwen25_coder_7b",   "ollama": "qwen2.5-coder:7b"},
    {"name": "deepseek_coder_6b", "ollama": "deepseek-coder:6.7b"},
    {"name": "codegemma_7b",      "ollama": "codegemma:7b"},
    {"name": "starcoder2_7b",     "ollama": "starcoder2:7b"},
    {"name": "codellama_7b",      "ollama": "codellama:7b-instruct"},
]

# Set True to only benchmark the first model — useful for a quick sanity check.
SINGLE_MODEL_SMOKE = False
if SINGLE_MODEL_SMOKE:
    OLLAMA_MODELS = OLLAMA_MODELS[:1]

print(f"Repo    : {UPSTREAM_REMOTE_URL} @ {UPSTREAM_REF}")
print(f"Results : {RESULTS_ROOT}")
print(f"Timeout : {MODEL_TIMEOUT}s per request")
print(f"Models  : {', '.join(m['ollama'] for m in OLLAMA_MODELS)}")


In [ ]:
#@title 🧑‍⚖️ Multi-LLM Judge — configure cloud key OR local model
# ──────────────────────────────────────────────────────────────────────────────
# The judge is SEPARATE from the Ollama model being benchmarked.
# Pick ONE of the two options below (or both — they are additive).
# ──────────────────────────────────────────────────────────────────────────────
import os
import subprocess

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  OPTION A — Cloud judge (Gemini, no VRAM cost)                             ║
# ║  Add "GEMINI_API_KEY" in Colab Secrets (🔑 left sidebar) to enable.        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
USE_CLOUD_JUDGE = True   # set False to disable cloud judge entirely

if USE_CLOUD_JUDGE:
    try:
        from google.colab import userdata
        _gemini = userdata.get("GEMINI_API_KEY")
        if _gemini:
            os.environ["GEMINI_API_KEY"] = _gemini
            print("✅ GEMINI_API_KEY loaded from Colab Secrets.")
        else:
            print("⚠  GEMINI_API_KEY not found in Colab Secrets — cloud judge disabled.")
            USE_CLOUD_JUDGE = False
    except Exception:
        print("ℹ  Not in Colab — skipping Secrets lookup for cloud judge.")
    # Uncomment to paste key directly (less secure):
    # os.environ["GEMINI_API_KEY"] = "YOUR_KEY_HERE"


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  OPTION B — Local Ollama judge (no API key, uses ~1 GB VRAM)               ║
# ║  A small dedicated model stays loaded alongside the benchmarked model.      ║
# ║  Recommended: qwen2.5:1.5b (~1 GB), phi3:mini (~2 GB), gemma2:2b (~1.5 GB) ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
USE_LOCAL_JUDGE  = True                   # set False to skip local judge
LOCAL_JUDGE_TAG  = "qwen2.5:1.5b"        # Ollama tag for the judge model
LOCAL_JUDGE_NAME = "local-judge"          # friendly name shown in results

if USE_LOCAL_JUDGE:
    os.environ["LOCAL_JUDGE_MODEL"] = LOCAL_JUDGE_TAG

    # Patch config.yaml at runtime to activate the local judge block
    import re
    from pathlib import Path

    cfg_path = Path("config.yaml")
    if cfg_path.exists():
        cfg_text = cfg_path.read_text()

        local_block = f"""
  - name: "{LOCAL_JUDGE_NAME}"
    provider: "local"
    endpoint: "http://localhost:11434/v1/chat/completions"
    model: "{LOCAL_JUDGE_TAG}"
    api_key: "not-needed"
    max_tokens: 500
    temperature: 0.1
    timeout: 120
"""
        # Inject the block after the judge_models: line if not already present
        if LOCAL_JUDGE_NAME not in cfg_text:
            cfg_text = cfg_text.replace(
                "judge_models:",
                "judge_models:" + local_block,
                1,
            )
            cfg_path.write_text(cfg_text)
            print(f"✅ Local judge '{LOCAL_JUDGE_NAME}' ({LOCAL_JUDGE_TAG}) injected into config.yaml")
        else:
            print(f"✅ Local judge '{LOCAL_JUDGE_NAME}' already present in config.yaml")

        # Pull the judge model now so it is ready before benchmarks start
        print(f"\nPulling judge model {LOCAL_JUDGE_TAG}...")
        subprocess.run(["ollama", "pull", LOCAL_JUDGE_TAG], check=True)
        print(f"✅ Judge model ready: {LOCAL_JUDGE_TAG}")
    else:
        print("⚠  config.yaml not found — run the clone-repo cell first.")
        USE_LOCAL_JUDGE = False


# ── Summary ───────────────────────────────────────────────────────────────────
active_judges = []
if USE_CLOUD_JUDGE and os.getenv("GEMINI_API_KEY"):
    active_judges.append("Gemini 2.0 Flash (cloud)")
if USE_LOCAL_JUDGE:
    active_judges.append(f"{LOCAL_JUDGE_TAG} (local Ollama)")

if active_judges:
    print(f"\n✅ Active judge(s): {', '.join(active_judges)}")
else:
    print("\n❌ No judges configured — llm_consensus_score will be 0.0 in results.")


In [ ]:
#@title 💾 Mount Google Drive (optional — skip if not needed)
# If Drive mounts successfully, RESULTS_ROOT is updated so results
# survive after the Colab session ends.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_ROOT = "/content/drive/MyDrive/HumanEvalComm_V2_Results"
    print(f"Drive mounted. Results → {RESULTS_ROOT}")
except Exception as e:
    print(f"Drive not mounted ({e}). Using local path: {RESULTS_ROOT}")


In [ ]:
#@title 📦 Clone / update repository
import os
import subprocess
from pathlib import Path


def run(cmd, cwd=None, check=True):
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=cwd, check=check, text=True,
                            capture_output=True)
    if result.stdout:
        print(result.stdout.rstrip())
    if result.stderr:
        print(result.stderr.rstrip())
    return result


repo_path = Path(REPO_DIR)
if repo_path.exists():
    print(f"Repo exists at {repo_path}. Fetching latest {UPSTREAM_REF}...")
    run(["git", "remote", "remove", "upstream"], cwd=repo_path, check=False)
    run(["git", "remote", "add",    "upstream", UPSTREAM_REMOTE_URL], cwd=repo_path)
    run(["git", "fetch", "--depth", "1", "upstream", UPSTREAM_REF], cwd=repo_path)
    run(["git", "checkout", "-B", "colab-run", "FETCH_HEAD"], cwd=repo_path)
else:
    run(["git", "init", str(repo_path)])
    run(["git", "remote", "add", "upstream", UPSTREAM_REMOTE_URL], cwd=repo_path)
    run(["git", "fetch", "--depth", "1", "upstream", UPSTREAM_REF], cwd=repo_path)
    run(["git", "checkout", "-B", "colab-run", "FETCH_HEAD"], cwd=repo_path)

os.chdir(repo_path)
run(["git", "rev-parse", "--short", "HEAD"], cwd=repo_path)
print(f"Working directory: {os.getcwd()}")


In [ ]:
#@title 🐍 Install Python dependencies
import sys
import subprocess
from pathlib import Path

req = Path("requirements_v2.txt")
if not req.exists():
    req = Path("requirement.txt")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
               check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "python-dotenv", "nbformat", "plotly", "kaleido", "statsmodels"],
    check=True,
)
print(f"✅ Dependencies installed from {req}")


In [ ]:
#@title 🦙 Install Ollama
import subprocess

print("Installing Ollama...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
               shell=True, check=True)
print("✅ Ollama installed.")


In [ ]:
#@title ▶️ Start Ollama server
import os
import subprocess
import time
import requests

os.environ["LOCAL_API_BASE"] = LOCAL_API_BASE
os.environ["OLLAMA_HOST"]    = "127.0.0.1:11434"

ollama_log = open("/tmp/ollama.log", "w")
subprocess.Popen(
    ["ollama", "serve"],
    stdout=ollama_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

print("Waiting for Ollama to be ready...")
ready = False
for _ in range(60):
    try:
        if requests.get("http://localhost:11434/api/tags", timeout=2).status_code == 200:
            ready = True
            break
    except Exception:
        pass
    time.sleep(2)

if ready:
    print("✅ Ollama server is ready.")
else:
    print("❌ Ollama did not start. Check /tmp/ollama.log:")
    print(open("/tmp/ollama.log").read())
    raise RuntimeError("Ollama startup failed")


In [ ]:
#@title ⬇️ Pull Ollama models
import subprocess

for m in OLLAMA_MODELS:
    print(f"\nPulling {m['ollama']}...")
    subprocess.run(["ollama", "pull", m["ollama"]], check=True)

print("\n✅ Models available:")
subprocess.run(["ollama", "list"], check=True)


In [ ]:
#@title 🔍 Pre-flight: verify Ollama API responds
# Sends one real chat request to the first model before the benchmark.
# If this fails, fix Ollama before proceeding.
import json
import requests

test_model = OLLAMA_MODELS[0]["ollama"]
print(f"Testing API with model: {test_model}")

resp = requests.post(
    "http://localhost:11434/v1/chat/completions",
    headers={"Content-Type": "application/json"},
    json={
        "model": test_model,
        "messages": [{"role": "user", "content": "Reply only with the word: OK"}],
        "max_tokens": 10,
        "temperature": 0.0,
    },
    timeout=MODEL_TIMEOUT,
)
resp.raise_for_status()
reply = resp.json()["choices"][0]["message"]["content"].strip()
print(f"✅ API response: {reply!r}")


In [ ]:
#@title 🧑‍⚖️ Pre-flight: verify multi-LLM judge API responds
# Sends one real scoring request to each configured judge model.
# If this fails, the benchmark will still run but judge scores will be 0.
import asyncio
import os
import sys

sys.path.insert(0, "src")

async def _test_judge():
    from evaluators.multi_llm_judge import MultiLLMJudge

    judge = MultiLLMJudge(config_path="config.yaml")
    if not judge.judge_models:
        print("⚠  No judge models found in config.yaml — skipping test.")
        return

    dummy_code    = "def add(a, b):\n    return a + b"
    dummy_problem = "Write a function that adds two numbers."
    dummy_expected = "Returns the sum of a and b."

    scores = await judge.evaluate_code(
        code=dummy_code,
        problem=dummy_problem,
        expected=dummy_expected,
    )
    if scores:
        print(f"✅ Judge working.")
        print(f"   Consensus score : {scores.consensus_score:.2f}")
        print(f"   Mean confidence : {scores.mean_confidence:.2f}")
        print(f"   Judges responded: {len(scores.judge_responses)}")
        for jr in scores.judge_responses:
            print(f"     • {jr.model_name}: score={jr.score:.1f}  conf={jr.confidence:.2f}")
    else:
        print("❌ Judge returned no scores — check API key and network access.")

asyncio.run(_test_judge())


In [ ]:
#@title 🔧 Build benchmark arguments + streaming helper
import os
import subprocess
from pathlib import Path

Path(RESULTS_ROOT).mkdir(parents=True, exist_ok=True)

# Format: name|ollama_tag|provider|max_tokens|temperature
# Using | as separator avoids ambiguity with colons inside Ollama model tags
# (e.g. "qwen2.5-coder:7b" contains a colon that would confuse : splitting).
MODEL_ARGS = []
for m in OLLAMA_MODELS:
    MODEL_ARGS.extend(["--models", f"{m['name']}|{m['ollama']}|local|1024|0.1"])

def run_benchmark(cmd, env=None):
    """Run a benchmark command and stream its stdout/stderr in real-time."""
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
        bufsize=1,
        universal_newlines=True,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

print("Models configured:")
for m in OLLAMA_MODELS:
    print(f"  {m['name']} → {m['ollama']}")
print(f"\nResults root : {RESULTS_ROOT}")
print(f"Model timeout: {MODEL_TIMEOUT}s")
print("run_benchmark() helper ready.")


In [ ]:
#@title 🧪 [SMOKE TEST] Run 2 problems — verify pipeline end-to-end
import os
from pathlib import Path

smoke_dir = Path(RESULTS_ROOT) / "smoke_test"
smoke_dir.mkdir(parents=True, exist_ok=True)

print(f"Running smoke test ({SMOKE_MAX_PROBLEMS} problems)...\n")

cmd = [
    "python3", "-u", "src/v2_benchmark.py",
    "--dataset-path",  "Benchmark/HumanEvalComm_v2.jsonl",
    "--api-provider",  "local",
    "--max-problems",  str(SMOKE_MAX_PROBLEMS),
    "--request-delay", "0",
    "--model-timeout", str(MODEL_TIMEOUT),
    "--output-dir",    str(smoke_dir),
    *MODEL_ARGS,
]

env = os.environ.copy()
env["LOCAL_API_BASE"] = LOCAL_API_BASE

run_benchmark(cmd, env=env)
print(f"\n✅ Smoke test done. Results → {smoke_dir}")


In [ ]:
#@title 📊 [FULL] Run standard V2 benchmark
import os
from pathlib import Path

standard_dir = Path(RESULTS_ROOT) / "standard_benchmark"
standard_dir.mkdir(parents=True, exist_ok=True)

print(f"Running full benchmark ({STANDARD_MAX_PROBLEMS} problems)...\n")

cmd = [
    "python3", "-u", "src/v2_benchmark.py",
    "--dataset-path",  "Benchmark/HumanEvalComm_v2.jsonl",
    "--api-provider",  "local",
    "--max-problems",  str(STANDARD_MAX_PROBLEMS),
    "--request-delay", "0",
    "--model-timeout", str(MODEL_TIMEOUT),
    "--output-dir",    str(standard_dir),
    *MODEL_ARGS,
]

env = os.environ.copy()
env["LOCAL_API_BASE"] = LOCAL_API_BASE

run_benchmark(cmd, env=env)
print(f"\n✅ Full benchmark done. Results → {standard_dir}")


In [ ]:
#@title 🚧 [PUSHBACK] Run unfeasible-requirements benchmark
import os
from pathlib import Path

unfeasible_dir = Path(RESULTS_ROOT) / "unfeasible_benchmark"
unfeasible_dir.mkdir(parents=True, exist_ok=True)

print(f"Running unfeasible benchmark ({UNFEASIBLE_MAX_PROBLEMS} problems)...\n")

cmd = [
    "python3", "-u", "src/v2_benchmark.py",
    "--dataset-path",  "Benchmark/HumanEvalComm_Unfeasible.jsonl",
    "--api-provider",  "local",
    "--max-problems",  str(UNFEASIBLE_MAX_PROBLEMS),
    "--request-delay", "0",
    "--model-timeout", str(MODEL_TIMEOUT),
    "--output-dir",    str(unfeasible_dir),
    *MODEL_ARGS,
]

env = os.environ.copy()
env["LOCAL_API_BASE"] = LOCAL_API_BASE

run_benchmark(cmd, env=env)
print(f"\n✅ Unfeasible benchmark done. Results → {unfeasible_dir}")


---

## 🔄 Per-Model Sequential Benchmark — GPU-Friendly

Run **one model at a time** to stay within Colab GPU memory limits.

For each model the pipeline:
1. **Pulls** the model from Ollama
2. Runs the **smoke test** (optional, 2 problems — quick sanity check)
3. Runs the **standard V2 benchmark** (full problem set)
4. Runs the **unfeasible / pushback benchmark**
5. **Deletes** the model from Ollama to reclaim VRAM before pulling the next one

### How the Multi-LLM Judge fits in

The judge and the benchmarked model are **completely independent**:

| | Benchmarked model | Multi-LLM Judge |
|---|---|---|
| **Runs on** | Ollama (local, GPU) | External cloud API (Gemini / OpenAI / Anthropic) |
| **VRAM used** | ~4 GB per model | **0 GB** — network call only |
| **Configured in** | `OLLAMA_MODELS` list | `judge_models` section of `config.yaml` |
| **API key needed** | None | `GEMINI_API_KEY` (or OpenAI / Anthropic) |

Because the judge calls a cloud API, it **works fine no matter which local Ollama model is currently loaded** — there is no conflict and no extra VRAM cost.

> **Prerequisite:** run the **🧑‍⚖️ Multi-LLM Judge API keys** cell (cell 3) and the
> **🧑‍⚖️ Pre-flight: verify judge** cell before starting benchmarks, so you know the
> judge is reachable before committing to a long run.

After all models finish, the **Merge** cell combines every per-model result into a single unified leaderboard.

> **Toggle options** in the next cell (`RUN_SMOKE`, `RUN_STANDARD`, `RUN_UNFEASIBLE`, `DELETE_AFTER_RUN`) to control what is executed.


In [ ]:
#@title 🔄 [PER-MODEL] Run each model individually — GPU-friendly
# ── Options ──────────────────────────────────────────────────────────────────
RUN_SMOKE        = True   # 2-problem sanity check before full runs
RUN_STANDARD     = True   # full standard V2 benchmark
RUN_UNFEASIBLE   = True   # unfeasible / pushback benchmark
DELETE_AFTER_RUN = True   # remove benchmarked model after each run (frees VRAM)
                          # NOTE: the local judge model is NOT deleted — it must
                          #       stay loaded to score the next model's outputs.
# ─────────────────────────────────────────────────────────────────────────────

import os
import subprocess
from pathlib import Path

PER_MODEL_ROOT = Path(RESULTS_ROOT) / "per_model"
PER_MODEL_ROOT.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["LOCAL_API_BASE"] = LOCAL_API_BASE

# Detect which model (if any) is used as a local judge so we never delete it
_judge_tag = os.getenv("LOCAL_JUDGE_MODEL", "")   # set by the judge-config cell

for m in OLLAMA_MODELS:
    alias = m["name"]
    tag   = m["ollama"]
    model_dir = PER_MODEL_ROOT / alias
    model_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*70}")
    print(f"  MODEL: {alias}  ({tag})")
    if _judge_tag:
        print(f"  JUDGE: {_judge_tag}  (local, stays loaded)")
    print(f"{'='*70}")

    # ── 1. Pull benchmarked model ─────────────────────────────────────────────
    print(f"\n[{alias}] ⬇  Pulling model...")
    subprocess.run(["ollama", "pull", tag], check=True)

    model_arg = f"{alias}|{tag}|local|1024|0.1"
    base_cmd  = [
        "python3", "-u", "src/v2_benchmark.py",
        "--api-provider",  "local",
        "--request-delay", "0",
        "--model-timeout", str(MODEL_TIMEOUT),
        "--models",        model_arg,
    ]

    # ── 2. Smoke test (optional) ──────────────────────────────────────────────
    if RUN_SMOKE:
        smoke_dir = model_dir / "smoke_test"
        smoke_dir.mkdir(parents=True, exist_ok=True)
        print(f"\n[{alias}] 🧪 Smoke test ({SMOKE_MAX_PROBLEMS} problems)...")
        run_benchmark(
            base_cmd + [
                "--dataset-path", "Benchmark/HumanEvalComm_v2.jsonl",
                "--max-problems", str(SMOKE_MAX_PROBLEMS),
                "--output-dir",   str(smoke_dir),
            ],
            env=env,
        )
        print(f"[{alias}] ✅ Smoke done → {smoke_dir}")

    # ── 3. Standard benchmark ─────────────────────────────────────────────────
    if RUN_STANDARD:
        std_dir = model_dir / "standard"
        std_dir.mkdir(parents=True, exist_ok=True)
        print(f"\n[{alias}] 📊 Standard benchmark ({STANDARD_MAX_PROBLEMS} problems)...")
        run_benchmark(
            base_cmd + [
                "--dataset-path", "Benchmark/HumanEvalComm_v2.jsonl",
                "--max-problems", str(STANDARD_MAX_PROBLEMS),
                "--output-dir",   str(std_dir),
            ],
            env=env,
        )
        print(f"[{alias}] ✅ Standard done → {std_dir}")

    # ── 4. Unfeasible / pushback benchmark ───────────────────────────────────
    if RUN_UNFEASIBLE:
        unf_dir = model_dir / "unfeasible"
        unf_dir.mkdir(parents=True, exist_ok=True)
        print(f"\n[{alias}] 🚧 Unfeasible benchmark ({UNFEASIBLE_MAX_PROBLEMS} problems)...")
        run_benchmark(
            base_cmd + [
                "--dataset-path", "Benchmark/HumanEvalComm_Unfeasible.jsonl",
                "--max-problems", str(UNFEASIBLE_MAX_PROBLEMS),
                "--output-dir",   str(unf_dir),
            ],
            env=env,
        )
        print(f"[{alias}] ✅ Unfeasible done → {unf_dir}")

    # ── 5. Free VRAM (benchmarked model only) ─────────────────────────────────
    if DELETE_AFTER_RUN:
        if tag == _judge_tag:
            print(f"\n[{alias}] ⚠  This model is also the judge — keeping it loaded.")
        else:
            print(f"\n[{alias}] 🗑  Removing benchmarked model to free VRAM...")
            subprocess.run(["ollama", "rm", tag], check=False)
            print(f"[{alias}] VRAM freed.  Judge model ({_judge_tag or 'cloud'}) still active.")

    print(f"\n[{alias}] ✅ All benchmarks done. Results → {model_dir}")

# ── Cleanup: remove judge model after all models are done ─────────────────────
if _judge_tag and DELETE_AFTER_RUN:
    print(f"\n🗑  All models done — removing judge model {_judge_tag}...")
    subprocess.run(["ollama", "rm", _judge_tag], check=False)
    print("Judge model removed.")

print(f"\n{'='*70}")
print(f"  ✅ All {len(OLLAMA_MODELS)} model(s) processed.")
print(f"  Results root: {PER_MODEL_ROOT}")
print(f"  Run the next cell to merge results into a combined leaderboard.")
print(f"{'='*70}")


In [ ]:
#@title 🗂️ Merge per-model results → combined leaderboards
# Reads each model's individual result files and produces one combined
# leaderboard CSV + JSON for each benchmark type (smoke / standard / unfeasible).

import glob
import json
import os
import pandas as pd
from pathlib import Path
from datetime import datetime
from IPython.display import display

PER_MODEL_ROOT = Path(RESULTS_ROOT) / "per_model"

def _latest_files(folder: Path, pattern: str):
    """Return all matching files in folder, sorted by name (timestamp)."""
    return sorted(folder.glob(pattern)) if folder.exists() else []

def merge_leaderboards(bench_subfolder: str) -> pd.DataFrame:
    """Combine the latest per-model leaderboard CSV for one benchmark type."""
    frames = []
    for model_dir in sorted(PER_MODEL_ROOT.iterdir()):
        if not model_dir.is_dir():
            continue
        csvs = _latest_files(model_dir / bench_subfolder, "v2_fixed_leaderboard_*.csv")
        if csvs:
            df = pd.read_csv(csvs[-1])
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def merge_results_json(bench_subfolder: str) -> list:
    """Combine the latest per-model results JSON for one benchmark type."""
    all_records = []
    for model_dir in sorted(PER_MODEL_ROOT.iterdir()):
        if not model_dir.is_dir():
            continue
        jsons = _latest_files(model_dir / bench_subfolder, "v2_fixed_results_*.json")
        if jsons:
            with open(jsons[-1]) as f:
                all_records.extend(json.load(f))
    return all_records

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
BENCH_TYPES = [
    ("smoke_test",  "SMOKE TEST"),
    ("standard",    "STANDARD BENCHMARK"),
    ("unfeasible",  "UNFEASIBLE PUSHBACK"),
]

for subfolder, label in BENCH_TYPES:
    df      = merge_leaderboards(subfolder)
    records = merge_results_json(subfolder)

    if df.empty:
        print(f"\n⚠  No results found for {label} — skipping.")
        continue

    out_csv  = PER_MODEL_ROOT / f"combined_leaderboard_{subfolder}_{ts}.csv"
    out_json = PER_MODEL_ROOT / f"combined_results_{subfolder}_{ts}.json"

    df.to_csv(out_csv, index=False)
    with open(out_json, "w") as f:
        json.dump(records, f, indent=2)

    print(f"\n{'='*60}")
    print(f"  {label}  ({len(df)} rows across {df['model_name'].nunique() if 'model_name' in df.columns else '?'} models)")
    print(f"{'='*60}")
    print(f"  CSV  → {out_csv.name}")
    print(f"  JSON → {out_json.name}")
    display(df)

print("\n✅ Merge complete. All combined files saved to:", PER_MODEL_ROOT)


In [ ]:
#@title 🏆 Display results leaderboards
import glob
import os
import pandas as pd
from pathlib import Path
from IPython.display import display

def show_leaderboard(title, folder):
    path = Path(RESULTS_ROOT) / folder
    files = glob.glob(str(path / "v2_fixed_leaderboard_*.csv"))
    print(f"\n{'='*60}")
    print(f"  {title}")
    print('='*60)
    if not files:
        print(f"  No results found in {folder}")
        return
    latest = max(files, key=os.path.getctime)
    print(f"  File: {os.path.basename(latest)}")
    df = pd.read_csv(latest)
    display(df)

show_leaderboard("SMOKE TEST",          "smoke_test")
show_leaderboard("STANDARD BENCHMARK",  "standard_benchmark")
show_leaderboard("UNFEASIBLE PUSHBACK",  "unfeasible_benchmark")


In [ ]:
#@title 📥 Zip and download results
import shutil
from pathlib import Path
from google.colab import files

archive = "/content/HumanEvalComm_V2_Ollama_Results"
shutil.make_archive(archive, "zip", RESULTS_ROOT)
print(f"Archive: {archive}.zip")

# Uncomment to trigger browser download:
# files.download(f"{archive}.zip")
